# HRP Database Setup

This notebook does the following:
1. Loading health monitoring data from Excel files
2. Setting up a local SQLite database for the data storage
3. Transforming and normalizing 17-sheets measurement data

**Summary of Results:**
- **Main Data File**: 17 parameter sheets (~1M rows each)
- **Medical Info**: 7,130 seniors with disease & medication data
- **SOS Alerts**: 1,092 alert records
- **Storage**: Local SQLite database

In [2]:
import sys
import os
import time
import sqlite3
from pathlib import Path
import warnings

import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.utils.database import initialize_database
from src.utils.load_data import load_all_data, print_summary_stats

warnings.filterwarnings('ignore')

## Section 1: Load and Inspect Raw Excel Data

Load the new collection: three measurement files, medical/diseases, seniors demographics (gender, birthdate, age), and SOS alerts. Inspect structure, types, and basic quality.

In [3]:
# Define paths
raw_data_dir = Path("../data/raw/HRP_new")
measurement_files = [
    raw_data_dir / "data_202512221122-01-09.xlsx",
    raw_data_dir / "data_202512221231-16-23.xlsx",
    raw_data_dir / "data_202512221344-24-30.xlsx",
]
med_file = raw_data_dir / "Med&Diseases_202512221410.xlsx"
demo_file = raw_data_dir / "SeniorGenderAge_202512221409.xlsx"
sos_file = raw_data_dir / "SOS_202512221411.xlsx"

# Ensure all files exist
for p in measurement_files + [med_file, demo_file, sos_file]:
    assert p.exists(), f"Missing file: {p}"

In [3]:
# Load Medications & Disease Data
df_medical = pd.read_excel(med_file, engine="openpyxl")
df_medical.shape

(8224, 3)

In [4]:
df_medical.columns

Index(['seniorID', 'diseaseNames', 'medicineNames'], dtype='object')

In [5]:
df_medical.dtypes

seniorID          int64
diseaseNames     object
medicineNames    object
dtype: object

In [6]:
df_medical.head()

,seniorID,diseaseNames,medicineNames
0,2875,"Osteoporoza,Nadciśnienie tętnicze,Arytmia serc...","Acard,Emanera,Agen,Concor,Valzek"
1,3755,"Miażdzyca,Osteoporoza","Gensulin,Beto,Furosemidum,Amlopin,Zahron,Berod..."
2,3762,"Cukrzyca,Niedoczynnośc tarczycy,Niedoczynnośc ...","Letrox,Diosminex,Valsacor,Metformax,Bibloc,Pol..."
3,3805,"Stomia,Niedosłuch,Skolioza","Pregabalin,Staveran,Neurovit"
4,4367,"Miażdżyca kończyn dolnych,Niewydolnośc układu ...","Allupol,Cipropol,Eliquis,Ezehron,Areplex"


In [7]:
df_medical.isnull().sum()

seniorID         0
diseaseNames     0
medicineNames    0
dtype: int64

In [8]:
# Load Demographics Data
df_demo_raw = pd.read_excel(demo_file, engine="openpyxl")
df_demo_raw.shape

(13317, 4)

In [9]:
df_demo_raw.columns

Index(['seniorID', 'gender', 'birthDate', 'age'], dtype='object')

In [10]:
df_demo_raw.dtypes

seniorID       int64
gender        object
birthDate     object
age          float64
dtype: object

In [11]:
df_demo_raw.isnull().sum()

seniorID       0
gender         0
birthDate    249
age          249
dtype: int64

In [12]:
# Load SOS Alerts
df_sos = pd.read_excel(sos_file, engine="openpyxl")
df_sos.shape

(5420, 3)

In [13]:
df_sos.columns

Index(['seniorID', 'alertDate', 'sosNote'], dtype='object')

In [14]:
df_sos.dtypes

seniorID              int64
alertDate    datetime64[ns]
sosNote              object
dtype: object

In [15]:
df_sos.head()

,seniorID,alertDate,sosNote
0,3205,2025-11-30 17:07:51,Alarm przypadkowy
1,3221,2025-11-25 19:38:36,Alarm przypadkowy
2,3275,2025-11-14 15:01:34,Alarm przypadkowy
3,3279,2025-11-09 11:58:31,Alarm przypadkowy
4,3283,2025-11-17 18:44:32,Alarm przypadkowy


In [16]:
df_sos.isnull().sum()

seniorID      0
alertDate     0
sosNote      85
dtype: int64

In [17]:
# Measurement data sheets (inspect first file)
xls = pd.ExcelFile(measurement_files[0])
sheet_names = xls.sheet_names
print(f"Total sheets in first file: {len(sheet_names)}")
print(f"Sheet names: {sheet_names}\n")

Total sheets in first file: 26
Sheet names: ['expdata', 'expdata#1', 'expdata#2', 'expdata#3', 'expdata#4', 'expdata#5', 'expdata#6', 'expdata#7', 'expdata#8', 'expdata#9', 'expdata#10', 'expdata#11', 'expdata#12', 'expdata#13', 'expdata#14', 'expdata#15', 'expdata#16', 'expdata#17', 'expdata#18', 'expdata#19', 'expdata#20', 'expdata#21', 'expdata#22', 'expdata#23', 'expdata#24', 'expdata#25']



In [18]:
for i, sheet_name in enumerate(sheet_names[:2]):
    df = pd.read_excel(measurement_files[0], sheet_name=sheet_name, nrows=5, engine="openpyxl")
    print(f"\nSheet '{sheet_name}':")
    print(f"    Col 0 (seniorID): {df.iloc[:, 0].values[:2]}")
    print(f"    Col 1 (value): {df.iloc[:, 1].values[:2]}")
    print(f"    Col 2 (sbp): {df.iloc[:, 2].values[:2]}")
    print(f"    Col 3 (dbp): {df.iloc[:, 3].values[:2]}")
    print(f"    Col 4 (date): {df.iloc[:, 4].values[:2]}")
    print(f"    Col 5 (type): {df.iloc[:, 5].values[:2]}")


Sheet 'expdata':
    Col 0 (seniorID): [48129 48427]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-03T23:16:12.000000000' '2025-11-03T23:16:12.000000000']
    Col 5 (type): ['Temperature' 'Temperature']

Sheet 'expdata#1':
    Col 0 (seniorID): [48313 42183]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-04T07:59:41.000000000' '2025-11-04T07:59:41.000000000']
    Col 5 (type): ['Temperature' 'Temperature']


## Section 2: Initialize SQLite Database

Create a local SQLite database with a schema for measurements, medical info, and alerts.

In [ ]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

# Close any prior connection to release file handle (Windows locks the file)
if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

# Delete existing database for fresh start
if db_path.exists():
    db_path.unlink()
    print("Deleted existing database")

In [33]:
# Create new database with schema
conn = initialize_database(db_path)
print(f"Database initialized at {db_path.absolute()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

INFO:src.utils.database:Database initialized at ..\db\hrp_data.db


Database initialized at c:\Users\eldar\Projects\AI-CVD\notebooks\..\db\hrp_data.db
Database size: 108.0 KB


In [34]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables created: {[t[0] for t in tables]}")


Tables created: ['seniors', 'measurements', 'sqlite_sequence', 'medical_info', 'diseases', 'medicines', 'senior_diseases', 'senior_medicines', 'alerts']


## Section 3: Store Data in SQLite Database

Use the reusable pipeline from `src.utils.load_data` to load demographics, measurements, medical info, and alerts into SQLite.

In [4]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

# Close any prior connection to release file handle (Windows locks the file)
if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

In [19]:
# Run end-to-end load using shared pipeline
# Use streaming to avoid large memory usage and commit in batches
# Set fresh_start=True to rebuild the DB and process all sheets from scratch
load_all_data(data_dir=raw_data_dir, fresh_start=True, streaming=True, batch_rows=100_000, resume=False)

INFO:src.utils.database:Database initialized at c:\Users\eldar\Projects\AI-CVD\db\hrp_data.db
INFO:src.utils.load_data:Loading seniors demographics from ..\data\raw\HRP_new\SeniorGenderAge_202512221409.xlsx


INFO:src.utils.load_data:Loaded 13317 senior demographic rows
INFO:src.utils.load_data:Upserted 13317 seniors with demographics
INFO:src.utils.load_data:Streaming measurements from ..\data\raw\HRP_new\data_202512221122-01-09.xlsx
INFO:src.utils.load_data:  expdata: +100,000 (total 100,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 200,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 300,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 400,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 500,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 600,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 700,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 800,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 900,000)
INFO:src.utils.load_data:  expdata: +100,000 (total 1,000,000)
INFO:src.utils.load_data:  expdata: +48,575 (total 1,048,575)
INFO:src.utils.load_data:✓ Finished sheet 'expdata'
INFO:src.utils.load_data:  expdata#1: +100,0


DATABASE SUMMARY
seniors.......................          13,317
measurements..................      72,654,799
alerts........................           5,420
medical_info (raw)............           8,220
diseases......................             161
medicines.....................           1,713
senior_diseases...............          49,444
senior_medicines..............          48,304


In [ ]:
# Re-open connection for downstream analysis
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Quick summary
print_summary_stats(conn)

In [ ]:
print(f"Database initialized at {db_path.absolute()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Pull measurements into pandas for inspection (sample only to avoid huge loads)
df_measurements = pd.read_sql("SELECT * FROM measurements LIMIT 100000", conn)
print(df_measurements.shape)
df_measurements.head()

In [ ]:
# Medical information counts
counts_med = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM medical_info", conn)
counts_med

Total records: 7130
Unique senior_ids: 7129


In [ ]:
# Alerts counts
counts_alerts = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM alerts", conn)
counts_alerts

Found 1 duplicate senior_ids - keeping last entry for each

Duplicate entries (showing all 2 rows):
      senior_id                                                                                                                                                                                                                       disease_names                                                                                  medicine_names
4842      46877  Nadciśnienie tętnicze,Arytmia serca,Zaćma,Dyskopatia,Zwyrodnienie kręgosłupa,Żylaki,Niedowidzenie,Jaskra,Miażdzyca,Osteoporoza,Niewydolnośc układu krążenia,Hipercholesterolemia,Migotanie przedsionków,Choroba wieńcowa,Reumatyzm  Betaserc,Coronal,Suvardio,Preductal,Cardilopin,Prestarium,Memotropil,Preductal,Cavinton,Tobrex
4843      46877  Nadciśnienie tętnicze,Arytmia serca,Zaćma,Dyskopatia,Zwyrodnienie kręgosłupa,Żylaki,Niedowidzenie,Jaskra,Miażdzyca,Osteoporoza,Niewydolnośc układu krążenia,Hipercholesterolemia,Migotanie przedsionków,Cho

In [ ]:
# Preview alerts
pd.read_sql("SELECT * FROM alerts LIMIT 5", conn)

Total alert records: 1092
Inserted 1092 alert records


In [ ]:
# Measurement type distribution
measure_type_counts = pd.read_sql(
    "SELECT type, COUNT(*) AS cnt FROM measurements GROUP BY type ORDER BY cnt DESC",
    conn,
)
measure_type_counts.head(20)

  [ 1/18] Loading 'expdata'... 1,048,575 rows
  [ 2/18] Loading 'expdata#1'... 1,048,575 rows
  [ 3/18] Loading 'expdata#2'... 1,048,575 rows
  [ 4/18] Loading 'expdata#3'... 1,048,575 rows
  [ 5/18] Loading 'expdata#4'... 1,048,575 rows
  [ 6/18] Loading 'expdata#5'... 1,048,575 rows
  [ 7/18] Loading 'expdata#6'... 1,048,575 rows
  [ 8/18] Loading 'expdata#7'... 1,048,575 rows
  [ 9/18] Loading 'expdata#8'... 

KeyboardInterrupt: 

In [ ]:
# Reload a manageable sample for detailed inspection
sample_measurements = pd.read_sql("SELECT * FROM measurements LIMIT 5000", conn)
sample_measurements.head()

(1048575, 6)

In [ ]:
sample_measurements.dtypes

,senior_id,value,sbp,dbp,date,type
0,36280,36.6,NaN,NaN,2025-11-13 09:17:37,Temperature
1,26847,36.6,NaN,NaN,2025-11-13 09:18:03,Temperature
2,47228,36.6,NaN,NaN,2025-11-13 09:18:10,Temperature
3,41278,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
4,18331,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
5,46340,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
6,12766,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
7,33548,36.4,NaN,NaN,2025-11-13 09:18:11,Temperature
8,9225,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
9,38226,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature


In [ ]:
sample_measurements['type'].unique()

senior_id             int64
value               float64
sbp                 float64
dbp                 float64
date         datetime64[ns]
type                 object
dtype: object

In [93]:
df_measurements['type'].unique()

array(['Temperature'], dtype=object)

In [ ]:
print("Data already loaded via load_all_data; batch insert step not needed here.")

Inserting measurements in batches...
  ✓ Inserted 500,000 measurements...
  ✓ Inserted 500,000 measurements...
  ✓ Inserted 1,000,000 measurements...
  ✓ Inserted 1,000,000 measurements...
  ✓ Inserted 1,500,000 measurements...
  ✓ Inserted 2,000,000 measurements...
  ✓ Inserted 2,500,000 measurements...
  ✓ Inserted 3,000,000 measurements...
  ✓ Inserted 3,500,000 measurements...
  ✓ Inserted 4,000,000 measurements...
  ✓ Inserted 4,500,000 measurements...
  ✓ Inserted 5,000,000 measurements...
  ✓ Inserted 5,500,000 measurements...
  ✓ Inserted 6,000,000 measurements...
  ✓ Inserted 6,500,000 measurements...
  ✓ Inserted 7,000,000 measurements...
  ✓ Inserted 7,500,000 measurements...
  ✓ Inserted 8,000,000 measurements...
  ✓ Inserted 8,500,000 measurements...
  ✓ Inserted 9,000,000 measurements...
  ✓ Inserted 9,500,000 measurements...
  ✓ Inserted 10,000,000 measurements...
  ✓ Inserted 10,500,000 measurements...
  ✓ Inserted 11,000,000 measurements...
  ✓ Inserted 11,500,000 meas

In [26]:
# Display database file size
db_size_mb = db_path.stat().st_size / (1024**2)
print(f"\nDB Size: {db_size_mb:.2f} MB")


DB Size: 3459.47 MB


In [ ]:
# Show Normalized tables
cursor = conn.cursor()
counts = {
    "seniors": cursor.execute("SELECT COUNT(*) FROM seniors").fetchone()[0],
    "measurements": cursor.execute("SELECT COUNT(*) FROM measurements").fetchone()[0],
    "alerts": cursor.execute("SELECT COUNT(*) FROM alerts").fetchone()[0],
    "medical_info": cursor.execute("SELECT COUNT(*) FROM medical_info").fetchone()[0],
    "diseases": cursor.execute("SELECT COUNT(*) FROM diseases").fetchone()[0],
    "medicines": cursor.execute("SELECT COUNT(*) FROM medicines").fetchone()[0],
    "senior_diseases": cursor.execute("SELECT COUNT(*) FROM senior_diseases").fetchone()[0],
    "senior_medicines": cursor.execute("SELECT COUNT(*) FROM senior_medicines").fetchone()[0],
}

print("\nRow counts after normalization:")
for table, count in counts.items():
    print(f"  {table:.<20} {count:>15,}")


Row counts after normalization:
  seniors.............          11,908
  measurements........      18,033,186
  alerts..............           1,092
  medical_info........           7,129
  diseases............             161
  medicines...........           1,626
  senior_diseases.....          41,879
  senior_medicines....          41,699


## Section 4: Verify Data Integrity

Check that all data was correctly transferred to the database and no data loss occurred.

In [57]:
# Verify row counts
cursor = conn.cursor()

counts = {
    "measurements": cursor.execute("SELECT COUNT(*) FROM measurements").fetchone()[0],
    "medical_info": cursor.execute("SELECT COUNT(*) FROM medical_info").fetchone()[0],
    "alerts": cursor.execute("SELECT COUNT(*) FROM alerts").fetchone()[0],
}

print("\n Row Count Verification:")
for table, count in counts.items():
    print(f"  {table:.<30} {count:>15,}")


 Row Count Verification:
  measurements..................      18,033,186
  medical_info..................           7,129
  alerts........................           1,092


In [58]:
# Verify unique seniors
print("\n Unique Seniors:")
unique_seniors = cursor.execute("SELECT COUNT(DISTINCT senior_id) FROM measurements").fetchone()[0]
print(f"  Total unique seniors: {unique_seniors:,}")


 Unique Seniors:
  Total unique seniors: 11,908
  Total unique seniors: 11,908


In [27]:
# Verify no NULL values in columns
print("\n NULL Value Checks:")
null_checks = {
    "measurements.senior_id": cursor.execute("SELECT COUNT(*) FROM measurements WHERE senior_id IS NULL").fetchone()[0],
    "measurements.date": cursor.execute("SELECT COUNT(*) FROM measurements WHERE date IS NULL").fetchone()[0],
    "measurements.type": cursor.execute("SELECT COUNT(*) FROM measurements WHERE type IS NULL").fetchone()[0],
}

for check, null_count in null_checks.items():
    status = "OK -" if null_count == 0 else "WARNING"
    print(f"  {status} {check:.<35} {null_count} NULLs")


 NULL Value Checks:
  OK - measurements.senior_id............. 0 NULLs
  OK - measurements.date.................. 0 NULLs
  OK - measurements.type.................. 0 NULLs


In [28]:
# Check data types and ranges
cursor.execute("""
    SELECT 
        type,
        COUNT(*) as count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        MIN(value) as min_val,
        MAX(value) as max_val,
        AVG(value) as avg_val
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY count DESC
""")

In [29]:
print("Data Distribution by Measurement Type")
for mtype, count, uniq_seniors, min_val, max_val, avg_val in cursor.fetchall():
    print(f"  {mtype:.<20} {count:>10,} rows | {uniq_seniors:>6,} seniors | {min_val:>8.2f} to {max_val:>8.2f} (avg: {avg_val:.2f})")

Data Distribution by Measurement Type
  Heartrate...........  4,074,735 rows | 11,762 seniors |    24.00 to   205.00 (avg: 73.42)
  Temperature.........  4,057,151 rows | 11,744 seniors |    36.30 to   127.90 (avg: 36.72)
  Saturation..........  3,250,993 rows | 11,708 seniors |    80.00 to    99.00 (avg: 97.08)
  Steps...............  2,575,576 rows | 10,168 seniors |     1.00 to 36955.00 (avg: 3173.54)


In [30]:
# Check blood pressure data
print("\nNo. of Rows with BP data:")
bp_count = cursor.execute("SELECT COUNT(*) FROM measurements WHERE sbp IS NOT NULL OR dbp IS NOT NULL").fetchone()[0]
bp_count


No. of Rows with BP data:


4074731

In [88]:
# Verify timestamp formats
print("\nDate/Timestamp Verification:")
cursor.execute("SELECT date FROM measurements LIMIT 5")
sample_timestamps = cursor.fetchall()
for ts in sample_timestamps:
    print(f"  Sample: {ts[0]}")


Date/Timestamp Verification:
  Sample: 2025-11-10 00:00:00
  Sample: 2025-11-10 00:00:01
  Sample: 2025-11-10 00:00:09
  Sample: 2025-11-10 00:00:10
  Sample: 2025-11-10 00:00:10


In [67]:
# Check date range
cursor.execute("SELECT MIN(date), MAX(date) FROM measurements")
min_date, max_date = cursor.fetchone()
print(f"\n  Date range: {min_date} to {max_date}")


  Date range: 2025-11-10 00:00:00 to 2025-11-15 23:59:56


In [31]:
# Check for duplicate measurements
duplicates = cursor.execute("""
    SELECT senior_id, date, type, COUNT(*) as dup_count
    FROM measurements
    GROUP BY senior_id, date, type
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchall()

In [32]:
print("\nDuplicate Check:")
if duplicates:
    print(f"  Found {len(duplicates)} potential duplicates (first 5):")
    for sid, date, mtype, count in duplicates:
        print(f"    senior_id={sid}, date={date}, type={mtype} appears {count} times")
else:
    print("  No duplicates were found")


Duplicate Check:
  Found 5 potential duplicates (first 5):
    senior_id=4442, date=2025-11-14 06:48:30, type=BloodPressure appears 2 times
    senior_id=4442, date=2025-11-14 06:48:30, type=Heartrate appears 2 times
    senior_id=8708, date=2025-11-13 06:07:11, type=BloodPressure appears 2 times
    senior_id=8708, date=2025-11-13 06:07:11, type=Heartrate appears 2 times
    senior_id=8709, date=2025-11-10 10:57:16, type=BloodPressure appears 2 times


## Section 5: Query and Validate Stored Data

Execute SQL queries to retrieve data and perform basic analysis to confirm database functionality.

### Example 1: Get all measurements for a specific type

In [78]:
# Example 1: Get Heartrate measurements (first 10) 
query1 = """
    SELECT senior_id, value, date, type
    FROM measurements
    WHERE type = 'Heartrate'
    ORDER BY date
    LIMIT 10
"""

In [79]:
df_example1 = pd.read_sql(query1, conn)
df_example1

,senior_id,value,date,type
0,11605,57.0,2025-11-10 00:00:10,Heartrate
1,49771,60.0,2025-11-10 00:00:10,Heartrate
2,29761,47.0,2025-11-10 00:00:10,Heartrate
3,30706,65.0,2025-11-10 00:00:10,Heartrate
4,50444,49.0,2025-11-10 00:00:10,Heartrate
5,43643,90.0,2025-11-10 00:00:10,Heartrate
6,38602,92.0,2025-11-10 00:00:10,Heartrate
7,22340,60.0,2025-11-10 00:00:10,Heartrate
8,23961,91.0,2025-11-10 00:00:10,Heartrate
9,6727,66.0,2025-11-10 00:00:10,Heartrate


### Example 2: Aggregate statistics by measurement type

In [80]:
# Example 2: Aggregate statistics by measurement type
query2 = """
    SELECT 
        type,
        COUNT(*) as measurement_count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        AVG(value) as avg_value,
        MIN(value) as min_value,
        MAX(value) as max_value,
        ROUND(AVG(value), 2) as mean
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY measurement_count DESC
"""

In [81]:
df_example2 = pd.read_sql(query2, conn)
df_example2

,type,measurement_count,unique_seniors,avg_value,min_value,max_value,mean
0,Heartrate,4074735,11762,73.424053,24.0,205.0,73.42
1,Temperature,4057151,11744,36.718686,36.3,127.9,36.72
2,Saturation,3250993,11708,97.076295,80.0,99.0,97.08
3,Steps,2575576,10168,3173.538838,1.0,36955.0,3173.54


### Example 3: Get measurements for a specific senior

In [94]:
# Find a sample senior ID first
sample_senior_id = int(df_measurements.iloc[0]["senior_id"])
query3 = """
    SELECT senior_id, value, sbp, dbp, date, type
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date DESC
    LIMIT 10
"""

In [95]:
df_example3 = pd.read_sql(query3, conn, params=[sample_senior_id])
df_example3

,senior_id,value,sbp,dbp,date,type
0,36280,99.0,NaN,NaN,2025-11-15 23:56:11,Saturation
1,36280,NaN,138.0,89.0,2025-11-15 23:56:11,BloodPressure
2,36280,64.0,NaN,NaN,2025-11-15 23:56:11,Heartrate
3,36280,36.6,NaN,NaN,2025-11-15 23:56:11,Temperature
4,36280,99.0,NaN,NaN,2025-11-15 23:46:11,Saturation
5,36280,NaN,134.0,89.0,2025-11-15 23:46:11,BloodPressure
6,36280,64.0,NaN,NaN,2025-11-15 23:46:11,Heartrate
7,36280,36.8,NaN,NaN,2025-11-15 23:46:11,Temperature
8,36280,99.0,NaN,NaN,2025-11-15 23:36:11,Saturation
9,36280,NaN,142.0,92.0,2025-11-15 23:36:11,BloodPressure


### Example 4: Query performance test

In [98]:
# Get all Heartrate measurements
start = time.time()
query4 = "SELECT * FROM measurements WHERE type = 'Heartrate' LIMIT 1000"
df_example4 = pd.read_sql(query4, conn)
elapsed = time.time() - start

In [99]:
print(f"  Retrieved {len(df_example4)} rows in {elapsed:.4f} seconds")

  Retrieved 1000 rows in 0.0254 seconds


In [100]:
df_example4.head(10)

,id,senior_id,value,sbp,dbp,date,type
0,6562468,11605,57.0,None,None,2025-11-10 00:00:10,Heartrate
1,6566465,49771,60.0,None,None,2025-11-10 00:00:10,Heartrate
2,6581644,29761,47.0,None,None,2025-11-10 00:00:10,Heartrate
3,6582055,30706,65.0,None,None,2025-11-10 00:00:10,Heartrate
4,6582056,50444,49.0,None,None,2025-11-10 00:00:10,Heartrate
5,6586749,43643,90.0,None,None,2025-11-10 00:00:10,Heartrate
6,6615195,38602,92.0,None,None,2025-11-10 00:00:10,Heartrate
7,6622152,22340,60.0,None,None,2025-11-10 00:00:10,Heartrate
8,6622153,23961,91.0,None,None,2025-11-10 00:00:10,Heartrate
9,6626011,6727,66.0,None,None,2025-11-10 00:00:10,Heartrate


### Example 4: Blood Pressure Analysis

In [102]:
query5 = """
    SELECT senior_id, sbp, dbp, date, type
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
    ORDER BY date DESC
    LIMIT 10
"""

In [103]:
df_example5 = pd.read_sql(query5, conn)
df_example5

,senior_id,sbp,dbp,date,type
0,9263,128.0,55.0,2025-11-15 23:59:53,BloodPressure
1,50494,112.0,89.0,2025-11-15 23:59:38,BloodPressure
2,26794,120.0,78.0,2025-11-15 23:59:33,BloodPressure
3,4043,151.0,57.0,2025-11-15 23:59:32,BloodPressure
4,48782,133.0,83.0,2025-11-15 23:59:32,BloodPressure
5,30983,126.0,72.0,2025-11-15 23:59:32,BloodPressure
6,49493,124.0,84.0,2025-11-15 23:59:31,BloodPressure
7,47346,119.0,75.0,2025-11-15 23:59:31,BloodPressure
8,21960,112.0,70.0,2025-11-15 23:59:31,BloodPressure
9,40467,132.0,75.0,2025-11-15 23:59:31,BloodPressure


### Example 5: Get Blood Pressure Statistics

In [104]:
query5_stats = """
    SELECT 
        COUNT(*) as bp_measurements,
        COUNT(DISTINCT senior_id) as seniors_with_bp,
        AVG(sbp) as avg_systolic,
        AVG(dbp) as avg_diastolic,
        MIN(sbp) as min_systolic,
        MAX(sbp) as max_systolic,
        MIN(dbp) as min_diastolic,
        MAX(dbp) as max_diastolic
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
"""

In [105]:
df_bp_stats = pd.read_sql(query5_stats, conn)
df_bp_stats

,bp_measurements,seniors_with_bp,avg_systolic,avg_diastolic,min_systolic,max_systolic,min_diastolic,max_diastolic
0,4074731,11762,129.392577,78.623184,68.0,212.0,23.0,156.0


## Section 6: Alert Quality Analysis

Identify and categorize technical/accidental/test/null alerts that should be excluded from analysis.

In [64]:
# Load all alerts from database
alerts_df = pd.read_sql("SELECT * FROM alerts", conn)

In [ ]:
len(alerts_df) # Number of alerts

1092

In [ ]:
alerts_df['senior_id'].nunique() # Unique seniors with alerts

821

In [61]:
alerts_df.head()

,senior_id,alert_date,sos_note,is_technical,alert_category
0,3275,2025-11-14 15:01:34,Alarm przypadkowy,True,TECHNICAL
1,3792,2025-11-12 11:17:06,Alarm przypadkowy,True,TECHNICAL
2,3794,2025-11-13 11:31:14,Alarm przypadkowy,True,TECHNICAL
3,3794,2025-11-13 11:10:03,Alarm przypadkowy,True,TECHNICAL
4,4365,2025-11-12 22:18:55,Alert techniczny,True,TECHNICAL


In [65]:
# Define patterns for technical/accidental/test alerts
exclusion_keywords = [
    'testowy',           # test
    'przypadkowy',       # accidental
    'techniczny',        # technical
    'test',
    'próba',             # attempt/trial
    'brak wskazań',      # no indication
    'bez wskazań'       # without indication
]

# Mark technical/test alerts
alerts_df['is_technical'] = alerts_df['sos_note'].fillna('').str.lower().apply(
    lambda x: any(keyword in x for keyword in exclusion_keywords)
)

In [68]:
# Create three alert categories: TECHNICAL, REAL (verified), UNVERIFIED
alerts_df['alert_category'] = 'UNVERIFIED'  # Default to unverified
alerts_df.loc[alerts_df['is_technical'], 'alert_category'] = 'TECHNICAL'  # Mark technical

# Mark real alerts (non-technical AND has description)
has_description = alerts_df['sos_note'].notna() & (alerts_df['sos_note'] != '')
alerts_df.loc[has_description & ~alerts_df['is_technical'], 'alert_category'] = 'REAL'

# Create separate dataframes for each category
technical_alerts_df = alerts_df[alerts_df['alert_category'] == 'TECHNICAL'].copy()
real_alerts_df = alerts_df[alerts_df['alert_category'] == 'REAL'].copy()
unverified_alerts_df = alerts_df[alerts_df['alert_category'] == 'UNVERIFIED'].copy()

In [69]:
# Alert statistics summary
print("\nALERT STATISTICS BY CATEGORY:\n")
for category, cat_df in [('TECHNICAL', technical_alerts_df), ('REAL', real_alerts_df), ('UNVERIFIED', unverified_alerts_df)]:
    pct = len(cat_df) / len(alerts_df) * 100
    unique_seniors = cat_df['senior_id'].nunique()
    print(f"{category:11s}: {len(cat_df):4d} alerts ({pct:5.1f}%) from {unique_seniors:3d} unique seniors")


ALERT STATISTICS BY CATEGORY:

TECHNICAL  : 1077 alerts ( 98.6%) from 813 unique seniors
REAL       :   13 alerts (  1.2%) from  13 unique seniors
UNVERIFIED :    2 alerts (  0.2%) from   2 unique seniors


In [74]:
# All unique alert messages with category labels
print("\nALL UNIQUE ALERT MESSAGES:\n")
all_messages = alerts_df['sos_note'].value_counts()

print(f"Total unique messages: {len(all_messages)}\n")

for msg, count in all_messages.items():
    category = alerts_df.loc[alerts_df['sos_note'] == msg, 'alert_category'].iat[0]
    status = '✓' if category == 'REAL' else ' '
    msg_preview = (msg or '<NULL>')[:85]
    print(f"{status} [{category:11}] {count:3} × {msg_preview}")


ALL UNIQUE ALERT MESSAGES:

Total unique messages: 8

  [TECHNICAL  ] 719 × Alarm przypadkowy
  [TECHNICAL  ] 234 × Alarm testowy
  [TECHNICAL  ]  36 × Alarm. Nie nawiązano kontaktu z Seniorem i opiekunem. W chwili obecnej brak wskazań d
  [TECHNICAL  ]  29 × Alarm. Nawiązano kontakt z Seniorem. W wyniku przeprowadzonej oceny stanu pacjenta ni
  [TECHNICAL  ]  29 × Alarm. Nie nawiązano kontaktu z Seniorem. Opiekun powiadomiony. W chwili obecnej brak
  [TECHNICAL  ]  16 × Alarm. Nawiązano kontakt z Seniorem. Opiekun powiadomiony. W chwili obecnej brak wska
  [TECHNICAL  ]  14 × Alert techniczny
✓ [REAL       ]  13 × Alarm. Nawiązano kontakt z Seniorem. W wyniku przeprowadzonej oceny stanu pacjenta wy
